In [1]:
import sys
sys.path.append("..")

import torch
import yaml
import importlib
import src.config as cfg
importlib.reload(cfg)
from src.config import *

from ultralytics import YOLO
from pathlib import Path

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\nDataset YAML    : {DATASET_YAML}")
print(f"Model base      : {MODEL_BASE}")
print(f"Epochs          : {EPOCHS}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Image size      : {IMG_SIZE}")

PyTorch version : 2.6.0+cu124
CUDA available  : True
GPU             : NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory      : 6.4 GB

Dataset YAML    : D:\drone-detection\dataset.yaml
Model base      : yolov8n.pt
Epochs          : 50
Batch size      : 16
Image size      : 640


In [2]:
gpu_mem = 0
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU Memory: {gpu_mem:.1f} GB")
print()

if gpu_mem >= 10:
    recommended = "yolov8m.pt"
    reason = ">=10GB GPU — medium model for best accuracy"
elif gpu_mem >= 6:
    recommended = "yolov8s.pt"
    reason = "6-10GB GPU — small model, good balance"
elif gpu_mem >= 4:
    recommended = "yolov8n.pt"
    reason = "4-6GB GPU — nano model, fast training"
else:
    recommended = "yolov8n.pt"
    reason = "CPU or low GPU — nano model only"

print(f"Recommended model : {recommended}")
print(f"Reason            : {reason}")
print()
print("You can override by changing MODEL_BASE in src/config.py")

GPU Memory: 6.4 GB

Recommended model : yolov8s.pt
Reason            : 6-10GB GPU — small model, good balance

You can override by changing MODEL_BASE in src/config.py


In [3]:
# This downloads the pretrained weights automatically (~6MB for nano)
model = YOLO(recommended)

print(f"Model loaded    : {recommended}")
print(f"Parameters      : {sum(p.numel() for p in model.model.parameters()):,}")
print(f"Task            : {model.task}")

Model loaded    : yolov8s.pt
Parameters      : 11,166,560
Task            : detect


In [ ]:
results = model.train(
    data        = str(DATASET_YAML),
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = 8,
    device      = 0,
    project     = str(MODELS_DIR / "runs"),
    name        = "visdrone_yolov8",
    exist_ok    = True,
    pretrained  = True,
    optimizer   = "AdamW",
    lr0         = 0.001,
    lrf         = 0.01,
    momentum    = 0.937,
    weight_decay= 0.0005,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,
    box         = 7.5,
    cls         = 0.5,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    degrees     = 0.0,
    translate   = 0.1,
    scale       = 0.5,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.1,
    cache       = "False",    # ← caches images in RAM, faster on laptops
    val         = True,
    save        = True,
    save_period = 10,
    plots       = True,
    verbose     = True,
)

print("\n✅ Training complete!")
print(f"Best weights saved at: {MODELS_DIR}/runs/visdrone_yolov8/weights/best.pt")

Ultralytics 8.4.50  Python-3.13.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\drone-detection\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=visdrone_yolov8, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, o

In [1]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory allocated : {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"GPU memory reserved  : {torch.cuda.memory_reserved(0)/1e9:.2f} GB")
print("GPU memory cleared!")

GPU memory allocated : 0.00 GB
GPU memory reserved  : 0.00 GB
GPU memory cleared!


In [2]:
import shutil, sys
sys.path.append("..")
from src.config import *

best_pt = MODELS_DIR / "runs" / "visdrone_yolov8" / "weights" / "best.pt"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(best_pt, WEIGHTS_DIR / "best.pt")
print(f"✅ Copied best.pt → {WEIGHTS_DIR / 'best.pt'}")
print(f"   Size: {(WEIGHTS_DIR / 'best.pt').stat().st_size / 1e6:.1f} MB")

✅ Copied best.pt → D:\drone-detection\models\weights\best.pt
   Size: 22.5 MB
